# Bloc 6 — Agrégation, GroupBy & Tableaux Croisés

Message clé : C'est le bloc qui transforme un 'manipulateur de données' en 'analyste'. groupby est l'outil Pandas le plus puissant.
Objectifs: comprendre Split-Apply-Combine, maîtriser groupby, agrégations multiples et nommées, transform vs agg, pivot_table et crosstab

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

**Expliquez ce que fait cette cellule avant de la lancer**

In [5]:
# 
sales = pd.read_csv('../01_data/exercises/thelook_sales.csv')

# 
sales['created_at'] = pd.to_datetime(sales['created_at'])

#
sales['mois'] = sales['created_at'].dt.month

# 
print(f"Dataset : {sales.shape[0]} ventes")

Dataset : 50000 ventes


## 1. Le paradigme Split-Apply-Combine
La fonction groupby divise les données en groupes (Split), applique une fonction à chacun (Apply), puis rassemble les résultats (Combine).
C'est l'équivalent du GROUP BY en SQL, mais en beaucoup plus flexible.

## 2. groupby() en pratique

### Un premier groupby

In [8]:
# Étape 1 : Calculer le prix moyen pour chaque catégorie (GroupBy + Mean)
prix_moyen_par_cat = sales.groupby('category')['sale_price'].mean()

# Étape 2 : Trier les résultats du plus cher au moins cher (Décroissant)
prix_moyen_sorted = prix_moyen_par_cat.sort_values(ascending=False)

# Étape 3 : Sélectionner le Top 10
top_10_plus_cheres = prix_moyen_sorted.head(10)


top_10_plus_cheres


category
Suits & Sport Coats              220.073391
Outerwear & Coats                173.215165
Blazers & Jackets                106.545893
Dresses                           82.271997
Jeans                             75.094493
Sweaters                          70.447753
Jumpsuits & Rompers               59.153557
Pants                             56.912311
Active                            52.979311
Fashion Hoodies & Sweatshirts     52.806003
Name: sale_price, dtype: float64

### 🧩 Comprendre `groupby()` grâce au parallèle avec SQL

Si vous connaissez déjà le langage **SQL**, le fonctionnement de `groupby()` dans Pandas est rigoureusement identique au **`GROUP BY`** des bases de données.

---

#### 1. Le mécanisme fondamental : *Split — Apply — Combine*

Sous le capot, Pandas comme SQL découpent le calcul en 3 temps :

```text
  [ Données Brutes ]
         │
         ▼ (1. SPLIT : Regrouper les lignes par clé)
 [ T-shirts ]   [ Pantalons ]   [ Chaussures ]
         │            │               │
         ▼ (2. APPLY : Appliquer la fonction d'agrégation, ex: AVG / .mean())
      [ 25.0€ ]    [ 60.0€ ]       [ 85.0€ ]
         │            │               │
         └────────────┼───────────────┘
                      ▼ (3. COMBINE : Assembler le résultat final)
               [ Tableau de synthèse ]
```

---

#### 2.  Tableau de correspondance Clause par Clause

| Étape de la requête | SQL | Pandas |
| :--- | :--- | :--- |
| **D'où vient la table ?** | `FROM sales` | `sales` |
| **Sur quoi regroupe-t-on ?** | `GROUP BY category` | `.groupby('category')` |
| **Sur quelle colonne calcule-t-on ?** | `SELECT sale_price` | `['sale_price']` |
| **Quelle formule applique-t-on ?** | `AVG(...)`, `SUM(...)`, `COUNT(*)` | `.mean()`, `.sum()`, `.count()` |
| **Comment trie-t-on ?** | `ORDER BY ... DESC` | `.sort_values(ascending=False)` |
| **Combien de lignes garde-t-on ?** | `LIMIT 10` | `.head(10)` |


> 💡 **À retenir :** 
> - En SQL, l'écriture commence par le `SELECT`, mais la base exécute d'abord le `FROM` puis le `GROUP BY`.
> - En Pandas, on écrit les étapes **dans l'ordre réel d'exécution** : de la table source (`sales`) jusqu'au résultat final (`head(10)`).
```

### Compter avec .size() ou .count()

In [9]:
# Combien de ventes par département ?
sales.groupby('department').size()

department
Men      24955
Women    25045
dtype: int64

### Plusieurs agrégations à la fois

In [ ]:
# Agrégations multiples sur sale_price
sales.groupby('category')['sale_price'].agg(['mean', 'sum', 'count']).head()

,mean,sum,count
category,,,
Accessories,40.660302,107749.80,2650
Active,52.979311,132289.34,2497
Blazers & Jackets,106.545893,285649.54,2681
Dresses,82.271997,213413.56,2594
Fashion Hoodies & Sweatshirts,52.806003,141625.70,2682


### Agrégations nommées (la syntaxe recommandée)

In [13]:
resultat = sales.groupby('category').agg(
    ca_total=('sale_price', 'sum'),
    nb_ventes=('sale_price', 'count'),
    prix_moyen=('sale_price', 'mean')
).sort_values('ca_total', ascending=False)

resultat.head(10)

,ca_total,nb_ventes,prix_moyen
category,,,
Suits & Sport Coats,568449.57,2583,220.073391
Outerwear & Coats,462138.06,2668,173.215165
Blazers & Jackets,285649.54,2681,106.545893
Dresses,213413.56,2594,82.271997
Jeans,193893.98,2582,75.094493
Jumpsuits & Rompers,193254.67,3267,59.153557
Sweaters,190913.41,2710,70.447753
Pants,169712.51,2982,56.912311
Fashion Hoodies & Sweatshirts,141625.70,2682,52.806003


### Grouper par plusieurs colonnes

In [11]:
# Un multi-index ! On verra comment l'aplatir.
sales.groupby(['department', 'category'])['sale_price'].mean().head()

department  category                     
Men         Accessories                       42.780145
            Active                            51.556036
            Blazers & Jackets                108.482167
            Dresses                           81.587410
            Fashion Hoodies & Sweatshirts     52.830641
Name: sale_price, dtype: float64

### Mini-exercice : à vous !
Quel est le chiffre d'affaires total par marque (brand) ?

In [15]:
sales.columns

Index(['id', 'order_id', 'user_id', 'product_id', 'sale_price', 'status',
       'created_at', 'shipped_at', 'delivered_at', 'returned_at',
       'product_name', 'category', 'brand', 'department', 'cost', 'gender',
       'age', 'country', 'city', 'traffic_source', 'mois'],
      dtype='str')

In [ ]:
# ca -> Chiffres d'affaires
ca_par_marque = sales.groupby('___')['___']."FONCTION"()
ca_par_marque.sort_values(ascending=False).head(10)

## 4. Tableaux croisés et pivot_table

### pivot_table : le fameux tableau croisé dynamique d'Excel !

In [ ]:
# L'équivalent d'un tableau croisé dynamique
pivot = pd.pivot_table(sales, values='sale_price', index='mois', columns='department', aggfunc='sum')
display(pivot)

### crosstab : compter les occurrences

In [ ]:
# Quelle est la différence avec pivot_table selon vous ?
display(pd.crosstab(sales['gender'], sales['traffic_source']))

---
# Exercices pratiques

## 🏋️ Exercice 6.1 — Le Tableau de Bord TheLook
1. CA total (somme de sale_price) par catégorie
2. Prix moyen par marque (top 15 par CA)
3. Nombre de ventes par mois — y a-t-il une tendance ?
4. Quelle catégorie a la marge totale la plus élevée (somme de sale_price - cost) ?
5. Visualisez un résultat avec .plot(kind='bar')

In [ ]:
# 1. CA par catégorie
# Votre code ici

In [ ]:
# 2. Prix moyen par marque (top 15)
# Votre code ici

In [ ]:
# 3. Ventes par mois
# Votre code ici

In [ ]:
# 4. Marge totale par catégorie
# Votre code ici

In [ ]:
# 5. Visualisation
# Votre code ici

## Exercice 6.2 — Le Profil Client
1. Groupez par gender ET traffic_source
2. Calculez : prix moyen, nombre de ventes, CA total (agrégations nommées)
3. Quel segment gender × traffic_source génère le plus de CA ?

In [ ]:
# 1 et 2. Groupby avec agrégations nommées
# Votre code ici

In [ ]:
# 3. Le segment le plus rentable
# Votre code ici

## Exercice 6.3 — Le Tableau Croisé de saison
1. Créez un pivot_table : mois en lignes, department en colonnes, somme de sale_price
2. Tracez le résultat en line chart (.plot(kind='line'))
3. Hommes et Femmes suivent-ils le même pattern saisonnier ?

In [ ]:
# 1. Pivot table
# Votre code ici

In [ ]:
# 2 et 3. Visualisation + analyse
# Votre code ici

---

## BONUS : Aller plus loin -> transform et filter
### transform : garder la taille du DataFrame original
agg() réduit les données (1 ligne par groupe), alors que transform() garde le même nombre de lignes.

In [ ]:
# Chaque ligne garde sa valeur + le total de son groupe
sales['ca_categorie'] = sales.groupby('category')['sale_price'].transform('sum')

sales['part_pct'] = (sales['sale_price'] / sales['ca_categorie'] * 100).round(2)

display(sales[['category', 'sale_price', 'ca_categorie', 'part_pct']].head())

### filter : éliminer des groupes entiers

utilisation de lambda function

In [ ]:
# filter() supprime des GROUPES entiers, pas des lignes individuelles
big_categories = sales.groupby('category').filter(lambda x: len(x) > 500)

print(f"Catégories conservées : {big_categories['category'].nunique()}")

### ⚖️ Différence majeure : `groupby().transform()` vs `groupby().filter()`

Après un `groupby()`, vous connaissez déjà `.agg()` (qui réduit chaque groupe à une seule ligne).  
**`.transform()`** et **`.filter()`** sont deux opérations avancées qui répondent à des logiques opposées :

| Méthode | Ce qu'elle fait | Nombre de lignes en sortie | Équivalent SQL |
| :--- | :--- | :---: | :--- |
| **`.transform()`** | Calcule une statistique par groupe et **la propage sur chaque ligne d'origine**. | **Identique** à la table d'origine | **Window Function**<br>`AVG(...) OVER (PARTITION BY ...)` |
| **`.filter()`** | Garde ou élimine des **groupes entiers** selon une condition globale sur le groupe. | **Inférieur ou égal** à la table d'origine | Clause **`HAVING`** |

---

### 1. `groupby().transform()` : Conserver la taille d'origine

`.transform()` renvoie une série qui a **exactement la même taille** que votre DataFrame initial. Elle permet d'attacher un indicateur de groupe directement à chaque ligne.

```text
Ligne 1 (T-shirt, 20€)   ──┐ GroupBy
Ligne 2 (T-shirt, 30€)   ──┤ Moyenne = 25€ ──► .transform() ──► [25€, 25€, 80€, 80€]
Ligne 3 (Jean, 70€)      ──┤ GroupBy
Ligne 4 (Jean, 90€)      ──┘ Moyenne = 80€

